# Association Rule Mining: Borrower Feature Patterns
**Issues: [R2-MX-01] #15 · [R2-MX-02] #16**

This notebook covers two tasks:
- **Issue #15 [R2-MX-01]** — Data preparation: discretize continuous borrower features into categorical bins for Apriori input.
- **Issue #16 [R2-MX-02]** — Apriori algorithm implementation: mine frequent itemsets, generate association rules, and interpret results using support, confidence, and lift.

---
**Data Source:** `data/borrower_profiles.csv`

---
## Part 1 — Issue #15: Association Rule Mining Data Preparation
### Step 1: Imports & Setup

In [ ]:
import sys, os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

sys.path.insert(0, os.path.dirname(os.getcwd()))

plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d2e',
    'axes.edgecolor':   '#3a3f5c',
    'axes.labelcolor':  '#e0e0f0',
    'xtick.color':      '#a0a8d0',
    'ytick.color':      '#a0a8d0',
    'text.color':       '#e0e0f0',
    'grid.color':       '#2a2f4a',
    'grid.linestyle':   '--',
    'grid.alpha':       0.5,
    'font.family':      'sans-serif',
    'figure.dpi':       120,
})

DATA_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')
PROC_DIR = os.path.join(DATA_DIR, 'processed')
print(f'Data dir: {DATA_DIR}')

### Step 2: Load Borrower Profiles

In [ ]:
bp = pd.read_csv(os.path.join(DATA_DIR, 'borrower_profiles.csv'))
print(f'Shape: {bp.shape}')
print(f'Columns: {bp.columns.tolist()}')
bp[['borrower_id', 'age', 'gender', 'location_type', 'education_level',
    'avg_monthly_income', 'gig_tenure_months', 'num_dependents',
    'loan_default_status']].head()

### Step 3: Discretize Continuous Borrower Features
Transform continuous features into meaningful categorical bins so they can be used as items in the Apriori algorithm.

In [ ]:
df = bp.copy()

# 1. Age: Young / Middle-Aged / Senior
df['age_group'] = pd.cut(
    df['age'],
    bins=[0, 30, 45, 100],
    labels=['age=Young(<=30)', 'age=Middle(31-45)', 'age=Senior(>45)']
)

# 2. Average Monthly Income: Low / Medium / High
df['income_group'] = pd.qcut(
    df['avg_monthly_income'],
    q=3,
    labels=['income=Low', 'income=Medium', 'income=High']
)

# 3. Gig Tenure: Beginner / Intermediate / Experienced
df['tenure_group'] = pd.cut(
    df['gig_tenure_months'],
    bins=[0, 12, 36, 1000],
    labels=['tenure=Beginner(<=12mo)', 'tenure=Intermediate(13-36mo)', 'tenure=Experienced(>36mo)']
)

# 4. Number of Dependents: None / Few / Many
df['dependents_group'] = pd.cut(
    df['num_dependents'],
    bins=[-1, 0, 2, 100],
    labels=['dependents=None', 'dependents=Few(1-2)', 'dependents=Many(>2)']
)

# 5. Income Volatility Index: Low / Medium / High
df['volatility_group'] = pd.qcut(
    df['income_volatility_index'],
    q=3,
    labels=['volatility=Low', 'volatility=Medium', 'volatility=High']
)

# 6. Savings Buffer Ratio: Low / Medium / High
df['savings_group'] = pd.qcut(
    df['savings_buffer_ratio'],
    q=3,
    labels=['savings=Low', 'savings=Medium', 'savings=High']
)

# 7. Debt-to-Inflow Ratio: Low / Medium / High
df['debt_group'] = pd.qcut(
    df['debt_to_inflow_ratio'],
    q=3,
    labels=['debt=Low', 'debt=Medium', 'debt=High']
)

# 8. Loan Default Status
df['default_label'] = df['loan_default_status'].map({0: 'default=No', 1: 'default=Yes'})

# 9. Categorical columns (already discrete)
df['gender_label']    = 'gender='    + df['gender'].astype(str)
df['location_label']  = 'location='  + df['location_type'].astype(str)
df['education_label'] = 'education=' + df['education_level'].astype(str)
df['platform_label']  = 'platform='  + df['primary_gig_platform'].astype(str)
df['bank_label']      = 'bank_acc='  + df['has_bank_account'].map({1: 'Yes', 0: 'No'})

print('Discretization complete. Value counts sample:')
print(df['income_group'].value_counts())
print(df['age_group'].value_counts())

### Step 4: Build Transaction-Format Dataset for Apriori
Each borrower becomes a "transaction" (basket) containing their feature labels as "items".

In [ ]:
# Columns to include as items
item_cols = [
    'age_group', 'income_group', 'tenure_group', 'dependents_group',
    'volatility_group', 'savings_group', 'debt_group', 'default_label',
    'gender_label', 'location_label', 'education_label', 'bank_label'
]

# Drop rows with any NaN in item_cols
df_clean = df[item_cols].dropna()
print(f'Records after dropping NaN: {len(df_clean)} / {len(df)}')

# Build list of transactions (each row is a list of items)
transactions = []
for _, row in df_clean.iterrows():
    transactions.append([str(v) for v in row.values])

print(f'\nSample transaction (first borrower):')
print(transactions[0])

### Step 5: One-Hot Encode Transactions (TransactionEncoder)

In [ ]:
te = TransactionEncoder()
te_array = te.fit_transform(transactions)
te_df = pd.DataFrame(te_array, columns=te.columns_)

print(f'Encoded matrix shape: {te_df.shape}')
print(f'Number of unique items: {te_df.shape[1]}')
te_df.head(3)

### Step 6: Save Prepared Association Data

In [ ]:
# Save the discretized (pre-encoding) dataset
df_clean.to_csv(os.path.join(PROC_DIR, 'association_rules_prepared.csv'), index=False)
print(f'Saved association_rules_prepared.csv  shape: {df_clean.shape}')

# Distribution plots for key bins
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Discretized Borrower Feature Distributions', fontsize=14, color='#e0e0f0')

plot_cols = ['age_group', 'income_group', 'tenure_group',
             'savings_group', 'debt_group', 'default_label']
colors = ['#7c83fd', '#fd7c83', '#83fdc5', '#ffd700', '#ff9f68', '#c083fd']

for ax, col, color in zip(axes.flat, plot_cols, colors):
    counts = df_clean[col].value_counts()
    ax.barh(counts.index.astype(str), counts.values, color=color, alpha=0.85, edgecolor='#3a3f5c')
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('Count')

plt.tight_layout()
plt.show()

---
## Part 2 — Issue #16: Apriori Implementation & Interpretation
### Step 7: Mine Frequent Itemsets with Apriori

In [ ]:
MIN_SUPPORT = 0.05  # at least 5% of borrowers

frequent_itemsets = apriori(
    te_df,
    min_support=MIN_SUPPORT,
    use_colnames=True,
    max_len=4
)
frequent_itemsets['length'] = frequent_itemsets['itemsets'].apply(len)
frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False).reset_index(drop=True)

print(f'Min support: {MIN_SUPPORT}')
print(f'Total frequent itemsets found: {len(frequent_itemsets)}')
print(f'\nItemsets by length:')
print(frequent_itemsets['length'].value_counts().sort_index().to_string())
print(f'\nTop 15 frequent itemsets:')
frequent_itemsets.head(15)

### Step 8: Support Distribution by Itemset Length

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Frequent Itemset Analysis', fontsize=14, color='#e0e0f0')

# Support distribution
length_colors = {1: '#7c83fd', 2: '#fd7c83', 3: '#83fdc5', 4: '#ffd700'}
for length, grp in frequent_itemsets.groupby('length'):
    axes[0].hist(grp['support'], bins=20, alpha=0.7,
                 color=length_colors.get(length, '#ccc'),
                 label=f'Length {length} ({len(grp)} itemsets)',
                 edgecolor='#3a3f5c')
axes[0].set_xlabel('Support')
axes[0].set_ylabel('Count')
axes[0].set_title('Support Distribution by Itemset Length')
axes[0].legend(framealpha=0.3)
axes[0].grid(True)

# Count by length
length_counts = frequent_itemsets['length'].value_counts().sort_index()
bars = axes[1].bar(length_counts.index.astype(str),
                   length_counts.values,
                   color=[length_colors.get(l, '#ccc') for l in length_counts.index],
                   edgecolor='#3a3f5c', alpha=0.85)
for bar, val in zip(bars, length_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 str(val), ha='center', va='bottom', fontsize=11, color='#e0e0f0')
axes[1].set_xlabel('Itemset Length')
axes[1].set_ylabel('Count')
axes[1].set_title('Number of Frequent Itemsets per Length')
axes[1].grid(True, axis='y')

plt.tight_layout()
plt.show()

### Step 9: Generate Association Rules (Support / Confidence / Lift)

In [ ]:
MIN_CONFIDENCE = 0.50
MIN_LIFT       = 1.0

rules = association_rules(
    frequent_itemsets,
    metric='confidence',
    min_threshold=MIN_CONFIDENCE,
    num_itemsets=len(frequent_itemsets)
)

# Apply lift filter
rules = rules[rules['lift'] >= MIN_LIFT].copy()

# Sort by lift descending
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

# Readable string versions
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(sorted(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(sorted(x)))

print(f'Min confidence: {MIN_CONFIDENCE}  |  Min lift: {MIN_LIFT}')
print(f'Total association rules generated: {len(rules)}')
rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']].head(20)

### Step 10: Focus on Default-Related Rules
Filter rules where the **consequent** is `default=Yes` or `default=No` — most actionable for credit risk.

In [ ]:
# Rules that predict default
default_rules = rules[
    rules['consequents_str'].isin(['default=Yes', 'default=No'])
].copy()

print(f'Rules predicting default outcome: {len(default_rules)}')
print(f"  -> default=Yes: {len(default_rules[default_rules['consequents_str']=='default=Yes'])}")
print(f"  -> default=No : {len(default_rules[default_rules['consequents_str']=='default=No'])}")

print('\nTop 20 Default-Related Rules (sorted by lift):')
default_rules[['antecedents_str', 'consequents_str', 'support', 'confidence', 'lift']].head(20)

### Step 11: Scatter Plot — Support vs Confidence (coloured by Lift)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Association Rules: Support / Confidence / Lift', fontsize=14, color='#e0e0f0')

# All rules
scatter = axes[0].scatter(
    rules['support'], rules['confidence'],
    c=rules['lift'], cmap='plasma',
    alpha=0.6, s=20, edgecolors='none'
)
plt.colorbar(scatter, ax=axes[0], label='Lift')
axes[0].set_xlabel('Support')
axes[0].set_ylabel('Confidence')
axes[0].set_title('All Rules (color=Lift)')
axes[0].grid(True)

# Default rules only - colour by consequent
yes_rules = default_rules[default_rules['consequents_str'] == 'default=Yes']
no_rules  = default_rules[default_rules['consequents_str'] == 'default=No']
axes[1].scatter(yes_rules['support'], yes_rules['confidence'],
                color='#fd7c83', alpha=0.75, s=40, label='default=Yes', edgecolors='none')
axes[1].scatter(no_rules['support'],  no_rules['confidence'],
                color='#83fdc5', alpha=0.75, s=40, label='default=No',  edgecolors='none')
axes[1].set_xlabel('Support')
axes[1].set_ylabel('Confidence')
axes[1].set_title('Default-Predicting Rules')
axes[1].legend(framealpha=0.3)
axes[1].grid(True)

for ax in axes:
    ax.set_facecolor('#1a1d2e')

plt.tight_layout()
plt.show()

### Step 12: Top Default-Predicting Rules — Horizontal Bar Chart

In [ ]:
TOP_N = 15

top_yes = default_rules[default_rules['consequents_str'] == 'default=Yes'].head(TOP_N)
top_no  = default_rules[default_rules['consequents_str'] == 'default=No'].head(TOP_N)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
fig.suptitle(f'Top {TOP_N} Association Rules by Lift', fontsize=14, color='#e0e0f0')

for ax, top_df, title, color in [
    (axes[0], top_yes, 'Rules => default=Yes (High Risk)', '#fd7c83'),
    (axes[1], top_no,  'Rules => default=No  (Low Risk)',  '#83fdc5')
]:
    labels = [f'{a} => {c}' for a, c in
              zip(top_df['antecedents_str'], top_df['consequents_str'])]
    # Wrap long labels
    labels = [l[:80] + '...' if len(l) > 80 else l for l in labels]

    bars = ax.barh(range(len(top_df)), top_df['lift'].values,
                   color=color, alpha=0.85, edgecolor='#3a3f5c')
    ax.set_yticks(range(len(top_df)))
    ax.set_yticklabels(labels, fontsize=7)
    ax.set_xlabel('Lift')
    ax.set_title(title, fontsize=10)
    ax.axvline(x=1.0, color='#ffd700', linestyle='--', linewidth=1.2, alpha=0.7, label='Lift=1 (baseline)')
    ax.legend(framealpha=0.3, fontsize=8)
    ax.grid(True, axis='x')
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

### Step 13: Rule Metrics Heatmap — Top Rules by Lift

In [ ]:
top20 = default_rules.head(20).copy()
top20['rule'] = [f"{a[:40]}... => {c}" if len(a) > 40 else f"{a} => {c}"
                 for a, c in zip(top20['antecedents_str'], top20['consequents_str'])]

heat_data = top20.set_index('rule')[['support', 'confidence', 'lift']].T

fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(
    heat_data, annot=True, fmt='.3f', cmap='YlOrRd',
    linewidths=0.5, ax=ax, cbar_kws={'shrink': 0.6}
)
ax.set_title('Top 20 Default-Predicting Rules: Support / Confidence / Lift', fontsize=12, pad=10)
ax.set_xticklabels(ax.get_xticklabels(), rotation=60, ha='right', fontsize=7)
plt.tight_layout()
plt.show()

### Step 14: Interpretation of Key Rules

In [ ]:
print('='*70)
print('  KEY RULE INTERPRETATIONS')
print('='*70)

top5_yes = default_rules[default_rules['consequents_str'] == 'default=Yes'].head(5)
top5_no  = default_rules[default_rules['consequents_str'] == 'default=No'].head(5)

print('\nHIGH RISK PATTERNS (=> default=Yes):')
for i, row in top5_yes.iterrows():
    print(f"\n  Rule: [{row['antecedents_str']}] => [default=Yes]")
    print(f"    Support    : {row['support']:.4f}  ({row['support']*100:.1f}% of borrowers)")
    print(f"    Confidence : {row['confidence']:.4f}  ({row['confidence']*100:.1f}% chance of default)")
    print(f"    Lift       : {row['lift']:.4f}  ({row['lift']:.1f}x more likely than baseline)")

print('\n' + '-'*70)
print('\nLOW RISK PATTERNS (=> default=No):')
for i, row in top5_no.iterrows():
    print(f"\n  Rule: [{row['antecedents_str']}] => [default=No]")
    print(f"    Support    : {row['support']:.4f}  ({row['support']*100:.1f}% of borrowers)")
    print(f"    Confidence : {row['confidence']:.4f}  ({row['confidence']*100:.1f}% chance of no default)")
    print(f"    Lift       : {row['lift']:.4f}  ({row['lift']:.1f}x more likely than baseline)")

### Step 15: Save All Rules to CSV

In [ ]:
export_cols = ['antecedents_str', 'consequents_str', 'support', 'confidence',
               'lift', 'leverage', 'conviction', 'zhangs_metric']

available_cols = [c for c in export_cols if c in rules.columns]
rules[available_cols].to_csv(
    os.path.join(PROC_DIR, 'association_rules_output.csv'), index=False
)
print(f'Saved association_rules_output.csv  ({len(rules)} rules)')

default_rules[available_cols].to_csv(
    os.path.join(PROC_DIR, 'association_rules_default_only.csv'), index=False
)
print(f'Saved association_rules_default_only.csv  ({len(default_rules)} rules)')

---
## Summary

### Issue #15 - Association Rule Mining Data Preparation
| Feature | Discretization |
|---------|----------------|
| `age` | Young (<=30) / Middle (31-45) / Senior (>45) |
| `avg_monthly_income` | Low / Medium / High (quantile thirds) |
| `gig_tenure_months` | Beginner (<=12mo) / Intermediate (13-36mo) / Experienced (>36mo) |
| `num_dependents` | None / Few (1-2) / Many (>2) |
| `income_volatility_index` | Low / Medium / High (quantile thirds) |
| `savings_buffer_ratio` | Low / Medium / High (quantile thirds) |
| `debt_to_inflow_ratio` | Low / Medium / High (quantile thirds) |
| `loan_default_status` | default=Yes / default=No |
| Categorical features | gender, location_type, education_level, has_bank_account |

### Issue #16 - Apriori Implementation & Interpretation
| Step | Description |
|------|-------------|
| 1 | Apriori mined frequent itemsets with min_support=5% |
| 2 | Association rules generated with min_confidence=50%, min_lift=1.0 |
| 3 | Default-predicting rules isolated (consequent = default=Yes/No) |
| 4 | Rules visualized via scatter plot (support vs confidence, coloured by lift) |
| 5 | Top rules plotted as horizontal bar charts by lift |
| 6 | Rule metrics heatmap produced for top 20 default-predicting rules |
| 7 | Detailed interpretation printed for top high-risk and low-risk patterns |
| 8 | All rules saved to `association_rules_output.csv` and `association_rules_default_only.csv` |